# Centauri Bloom — Unified trainer
**One trainer. One model. Multiple datasets.** Select a GPU, choose a dataset and steps, confirm data permission, run setup, authorize, then run training and upload. Defaults: Jaron, 3,000 steps. Jaron uses response-only instruction tuning; text corpora use next-token pretraining. Raw contributor data stays private in Drive. All runs continue the newest merged compatible model and submit a reviewable PR, never a direct main commit. A completed run can be uploaded later with [upload recovery](https://colab.research.google.com/github/InterstellarNerd8760/centauri-bloom/blob/main/notebooks/bloom-upload-recovery.ipynb). Recovery of processed windows is supported before the first checkpoint; automatic partial optimizer resume is not implemented. Repeated exposure to a small corpus may overfit. Use reviewed data and authorized hardware.

In [ ]:
#@title 1. Choose training settings
DATASET = "Jaron" #@param ["Jaron","FineWeb-Edu + TinyStories","FineWeb-Edu","TinyStories","Cosmopedia","Public-domain literature","Public-domain + TinyStories","Arithmetic instruction pilot"]
STEPS = 3000 #@param [500,1000,1500,3000,5000,10000] {type:"raw"}
#@markdown Review the selected dataset's terms and confirm you may train on it and submit candidate weights.
CONFIRM_DATA_PERMISSION = False #@param {type:"boolean"}
#@markdown Optional: recover processed windows from a previous unified run before its first checkpoint.
RECOVER_RUN_DIR = "" #@param {type:"string"}
if not CONFIRM_DATA_PERMISSION:
    raise ValueError("Confirm dataset permission before running.")
print("Selected:",DATASET,"|",STEPS,"optimizer step attempts")


In [ ]:
#@title 2. Set up GPU and Google Drive
import os,sys,subprocess,json
from pathlib import Path
from datetime import datetime,timezone
from google.colab import drive
os.environ['HF_HUB_DISABLE_XET']='1'
REPO=Path('/content/centauri-bloom')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/InterstellarNerd8760/centauri-bloom.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'],check=True)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','tokenizers','datasets','huggingface_hub','ipywidgets'],check=True)
subprocess.run([sys.executable,'-c','from bloom.notebook_workflow import preflight; preflight("cuda")'],check=True)
from bloom.notebook_workflow import preflight,submit_candidate,sha256
from bloom.trainer import run_training,REPO_ID
from bloom.trainer_data import JARON_SHA,RECIPES
preflight('cuda')
if DATASET not in RECIPES or type(STEPS) is not int or not 1<=STEPS<=10000:
    raise ValueError('Invalid dataset or step budget.')
import torch
print('GPU:',torch.cuda.get_device_name(0))
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/centauri-bloom-runs')
BASE.mkdir(parents=True,exist_ok=True)
slug=DATASET.lower().replace(' + ','-').replace(' ','-')
OUT_DIR=Path(RECOVER_RUN_DIR) if RECOVER_RUN_DIR else BASE/('bloom-'+slug+'-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
HF_REPO_ID=REPO_ID
SUBMIT_HF_PR=True
print('Output:',OUT_DIR)


In [ ]:
#@title 3. Authorize Hugging Face PR
from huggingface_hub import HfApi
from bloom.notebook_workflow import auth_client
AUTH_API = None
if not SUBMIT_HF_PR:
    AUTH_API = HfApi()
    print('Local-only run: no upload.')
elif IN_COLAB:
    from IPython.display import display
    import ipywidgets as widgets
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    token_field=widgets.Password(description='HF token:',placeholder='Paste token')
    authorize=widgets.Button(description='Authorize upload',button_style='success')
    auth_out=widgets.Output()
    def authenticate(_):
        global AUTH_API
        with auth_out:
            auth_out.clear_output()
            try:
                AUTH_API=auth_client(HF_REPO_ID,token_field.value)
                token_field.value=''
                authorize.disabled=True
                print('Authenticated. Candidate PR only. HF checks write permissions on submission.')
            except Exception as exc:
                print('Authentication failed:',type(exc).__name__)
    authorize.on_click(authenticate)
    display(token_field,authorize,auth_out)
else:
    from getpass import getpass
    token=os.environ.get('HF_TOKEN')
    if not token and IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            pass
    AUTH_API=auth_client(HF_REPO_ID,token or getpass('HF token: '))
    del token


In [ ]:
#@title 4. Load latest model, prepare selected data, train and evaluate
if AUTH_API is None:
    raise RuntimeError('Tap Authorize upload in the preceding cell.')
jaron_path=None
if DATASET=='Jaron' and not RECOVER_RUN_DIR:
    folder=Path('/content/drive/MyDrive/centauri-bloom-data')
    folder.mkdir(parents=True,exist_ok=True)
    jaron_path=folder/'bloom_instruction_expanded.json'
    if not jaron_path.exists():
        from google.colab import files
        print('Choose the original bloom_instruction_expanded.json.')
        uploaded=files.upload()
        raw=uploaded.get(jaron_path.name)
        if raw is None:
            raise ValueError('Select the original contributor JSON.')
        import hashlib
        if hashlib.sha256(raw).hexdigest()!=JARON_SHA:
            raise ValueError('Contributor file hash mismatch.')
        jaron_path.write_bytes(raw)
    if sha256(jaron_path)!=JARON_SHA:
        raise ValueError('Contributor file hash mismatch.')
metrics=run_training(AUTH_API,DATASET,STEPS,REPO,OUT_DIR,jaron_path=jaron_path,recover=bool(RECOVER_RUN_DIR))


In [ ]:
#@title 5. Automatically submit the completed candidate
if AUTH_API is None:
    raise RuntimeError('Authorize upload first.')
metrics=json.loads((OUT_DIR/'metrics.json').read_text())
if metrics['additional_steps']!=STEPS or metrics['dataset']!=DATASET:
    raise RuntimeError('Run settings differ from completed candidate.')
submit_candidate(AUTH_API,HF_REPO_ID,OUT_DIR,STEPS,metrics['parent_revision'],metrics['run_name'])
print('Review and merge the candidate in Hugging Face before the next run.')
